In [2]:
from datasets import load_dataset
import torch
import unicodedata
from pypinyin import lazy_pinyin
import re
from torchaudio.pipelines import MMS_FA as bundle
from typing import List
import IPython
from tqdm.auto import tqdm

In [3]:
dataset = load_dataset("CAiRE/ASCEND")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

README.md: 0.00B [00:00, ?B/s]

main/train-00000-of-00003.parquet:   0%|          | 0.00/317M [00:00<?, ?B/s]

main/train-00001-of-00003.parquet:   0%|          | 0.00/367M [00:00<?, ?B/s]

main/train-00002-of-00003.parquet:   0%|          | 0.00/328M [00:00<?, ?B/s]

main/test-00000-of-00001.parquet:   0%|          | 0.00/106M [00:00<?, ?B/s]

main/validation-00000-of-00001.parquet:   0%|          | 0.00/107M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/9869 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1315 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1130 [00:00<?, ? examples/s]

cuda


In [4]:
def is_chinese(char):
    return 'CJK UNIFIED IDEOGRAPH' in unicodedata.name(char, '')

def is_english(char):
    return char.isascii() and char.isalpha()

def tokenize_with_languages(text):
    tokens = []
    token_languages = []
    current = ""
    for char in text:
        if is_chinese(char):
            if current:
                tokens.append(current)
                token_languages.append("en")
                current=""
            tokens.append(char)
            token_languages.append("zh")
        elif is_english(char):
            current += char
        elif char==" " and current:
            tokens.append(current)
            token_languages.append("en")
            current=""
    if current:
        tokens.append(current)
        token_languages.append("en")
    return tokens, token_languages

In [5]:
def romanize(text):
    return " ".join(lazy_pinyin(text))

In [6]:
def normalize(text):
    text = text.lower()
    text = text.replace("’", "'")
    text = re.sub("([^a-z' ])", " ", text)
    text = re.sub(' +', ' ', text)
    return text.strip()

In [7]:
model = bundle.get_model()
model.to(device)

tokenizer = bundle.get_tokenizer()
aligner = bundle.get_aligner()

Downloading: "https://dl.fbaipublicfiles.com/mms/torchaudio/ctc_alignment_mling_uroman/model.pt" to /root/.cache/torch/hub/checkpoints/model.pt


100%|██████████| 1.18G/1.18G [00:03<00:00, 402MB/s] 


In [8]:
print(bundle.get_dict())

{'-': 0, 'a': 1, 'i': 2, 'e': 3, 'n': 4, 'o': 5, 'u': 6, 't': 7, 's': 8, 'r': 9, 'm': 10, 'k': 11, 'l': 12, 'd': 13, 'g': 14, 'h': 15, 'y': 16, 'b': 17, 'p': 18, 'w': 19, 'c': 20, 'v': 21, 'j': 22, 'z': 23, 'f': 24, "'": 25, 'q': 26, 'x': 27, '*': 28}


In [9]:
def compute_alignments(waveform: torch.Tensor, transcript: List[str]):
    with torch.inference_mode():
        emission, _ = model(waveform.to(device))
        token_spans = aligner(emission[0], tokenizer(transcript))
    return emission, token_spans

In [10]:
def _score(spans):
    return sum(s.score * len(s) for s in spans) / sum(len(s) for s in spans)

In [11]:
def preview_word(token_language, waveform, spans, num_frames, transcript, sample_rate=bundle.sample_rate):
    ratio = waveform.size(1) / num_frames
    x0 = int(ratio * spans[0].start)
    x1 = int(ratio * spans[-1].end)
    print(f"{token_language} - {transcript} ({_score(spans):.2f}): {x0 / sample_rate:.3f} - {x1 / sample_rate:.3f} sec")
    segment = waveform[:, x0:x1]
    return IPython.display.Audio(segment.numpy(), rate=sample_rate)

In [12]:
def code_switch(waveform, transcript, token_languages, token_spans, num_frames, sample_rate):
    switches = []
    
    samples_per_frame = waveform.size(1) / num_frames

    for i in range(1, len(transcript)):
        previous_language = token_languages[i - 1]
        current_language = token_languages[i]

        if previous_language != current_language:
            switch_time = (samples_per_frame * token_spans[i][0].start) / sample_rate
            
            switches.append({"switch": round(switch_time, 3), "from": previous_language, "to": current_language})
    
    return switches

In [13]:
sample = dataset["train"][2]
print(sample)

{'id': '00002', 'path': '/storage/hf-datasets-cache/all/datasets/16739474757983-config-parquet-and-info-CAiRE-ASCEND-5c1abf9c/downloads/extracted/f0790e45797bd654a35ecd1eb4865fa761f1cbd842b674e0defb6812ae8cffbf/waves/ses1_spk1_L6_6.720_3.320.wav', 'audio': <datasets.features._torchcodec.AudioDecoder object at 0x7bc1fbaa7830>, 'transcription': '嗯初次见面nice to meet you嗯', 'duration': 3.319999933242798, 'language': 'mixed', 'original_speaker_id': 1, 'session_id': 1, 'topic': 'persona'}


In [14]:
text_raw = sample["transcription"]
tokens, token_languages = tokenize_with_languages(text_raw)
text_romanized = romanize(" ".join(tokens))
text_normalized = normalize(text_romanized)
transcript = text_normalized.split()

print("Tokens:", tokens)
print("Languages:", token_languages)
print("Romanized transcript:", text_romanized)
print("Normalized transcript:", text_normalized)

waveform = torch.tensor(sample["audio"]["array"], dtype=torch.float32).unsqueeze(0)
sample_rate = sample["audio"]["sampling_rate"]
print("Audio:")
display(IPython.display.Audio(waveform, rate=sample_rate))

emission, token_spans = compute_alignments(waveform, transcript)
num_frames = emission.size(1)
for i in range(len(transcript)):
    display(preview_word(token_languages[i], waveform, token_spans[i], num_frames, transcript[i]))

switches = code_switch(waveform, transcript, token_languages, token_spans, num_frames, sample_rate)
for switch in switches:
    print(switch)

Tokens: ['嗯', '初', '次', '见', '面', 'nice', 'to', 'meet', 'you', '嗯']
Languages: ['zh', 'zh', 'zh', 'zh', 'zh', 'en', 'en', 'en', 'en', 'zh']
Romanized transcript: n   chu   ci   jian   mian  nice to meet you  n
Normalized transcript: n chu ci jian mian nice to meet you n
Audio:


zh - n (0.02): 0.060 - 0.080 sec


zh - chu (0.73): 0.443 - 0.543 sec


zh - ci (0.49): 0.624 - 0.704 sec


zh - jian (0.72): 0.744 - 0.905 sec


...

en - to (0.14): 1.549 - 1.630 sec


en - meet (0.30): 1.690 - 1.871 sec


en - you (0.07): 1.932 - 2.072 sec


zh - n (0.04): 2.374 - 2.394 sec


{'switch': 1.288, 'from': 'zh', 'to': 'en'}
{'switch': 2.374, 'from': 'en', 'to': 'zh'}


In [15]:
def get_switches(sample):
    text_raw = sample["transcription"]
    tokens, token_languages = tokenize_with_languages(text_raw)
    text_romanized = romanize(" ".join(tokens))
    text_normalized = normalize(text_romanized)
    transcript = text_normalized.split()
    if len(transcript) != len(token_languages):
        raise ValueError(f"Token mismatch: {len(transcript)} transcript tokens vs {len(token_languages)} language labels")

    waveform = torch.tensor(sample["audio"]["array"], dtype=torch.float32).unsqueeze(0)
    sample_rate = sample["audio"]["sampling_rate"]
    if sample_rate != bundle.sample_rate:
        raise ValueError(f"Expected sample rate {bundle.sample_rate} got {sample_rate}")

    emission, token_spans = compute_alignments(waveform, transcript)
    num_frames = emission.size(1)

    switches = code_switch(waveform, transcript, token_languages, token_spans, num_frames, sample_rate)

    return switches

In [16]:
all_switches = {}
failures = []

for split in dataset.keys():

    all_switches[split] = []

    for idx, sample in enumerate(tqdm(dataset[split], desc=split)):

        if sample["language"] != "mixed":
            continue

        if "[UNK]" in sample["transcription"]:
            continue

        try:
            switches = get_switches(sample)

            all_switches[split].append({
                "index": idx,
                "duration": len(sample["audio"]["array"]) / sample["audio"]["sampling_rate"],
                "num_switches": len(switches),
                "switches": switches
            })

        except Exception as e:
            failures.append({
                "split": split,
                "index": idx,
                "error": str(e)
            })

train:   0%|          | 0/9869 [00:00<?, ?it/s]

test:   0%|          | 0/1315 [00:00<?, ?it/s]

validation:   0%|          | 0/1130 [00:00<?, ?it/s]

In [17]:
import json

with open("code_switch_timestamps.json", "w") as f:
    json.dump(all_switches, f, indent=2)

with open("failures.json", "w") as f:
    json.dump(failures, f, indent=2)